# 使用音频说话人分离构建能识别说话人的会议智能流水线

许多组织已经在录制重要的对话，但单纯的文字稿往往不足以支撑可靠的后续跟进。缺失的一环是说话人归属：知道是谁提出了顾虑、是谁做出了承诺，以及证据出现在录音的哪个位置。能识别说话人的文字稿可以让你把客户需求与销售方的跟进区分开，把结构化的证据引用与行动项放在一起，并在敏感承诺进入 CRM、工单系统或知识库之前，将其转入审核工作流。

只要下游工作流取决于“谁说了什么”，这种模式就很有用：

- 销售探索与解决方案咨询：带证据地记录客户需求、销售方承诺、决策标准、阻碍因素和后续步骤。
- 客户成功与客户管理：将 QBR、续约电话和入门引导会议转化为有出处的风险、产品需求和跟进计划。
- 支持升级与事故复盘：在创建工单或事后复盘之前，保留时间线、所报告的症状、负责人承诺以及未解决的问题。
- 招聘与面试流程：总结候选人或面试官的反馈，同时让引语与正确的说话人保持关联。
- 受监管或高风险的审核：在存储来自医疗、金融服务、法律或合规要求严格的对话笔记之前，加入脱敏、证据检查和人工审核。

对于营收团队而言，其价值通常不在于再生成一份摘要，而在于减少从对话到记录系统之间的信息流失。这样的流水线可以帮助团队更快地记录可直接写入 CRM 的后续步骤，更早地识别续约或扩展风险，为预测更新保留背后的证据，并基于有出处的实例而不是道听途说的笔记来辅导销售代表或支持团队。目标不是用自动化取代判断，而是让交接、审核和后续行动更完整、更可审计。

本 notebook 展示如何使用 OpenAI 音频说话人分离（diarization）构建一个生产风格的通话后会议智能流水线。你将：

1. 接收一个已录制的会议音频文件。
2. 可选地使用简短的参考音频片段来映射已知说话人。
3. 调用 `gpt-4o-transcribe-diarize`，并设置 `response_format="diarized_json"`。
4. 将带说话人标签的片段规范化为带有稳定片段 ID 的 JSON 和 Markdown。
5. 使用结构化输出提取会议简报、决策、风险、明确提出的问题、建议的跟进事项、行动项、证据引用以及一份跟进邮件草稿。
6. 写出可供审核的产物以及一份本地防护（guardrail）报告。

默认单元格使用一份合成的说话人分离文字稿运行，无需 API 密钥。真实音频调用需要主动开启，因此可以放心地从头到尾审阅本 notebook。


## 架构

![架构图](images/architecture.svg)

| 层 | 职责 | 输出 |
| --- | --- | --- |
| 音频接入 | 接收通话录音以及可选的已知说话人片段。 | `meeting.wav`、`Agent=agent.wav` |
| 流水线运行器 | 校验输入、将参考片段编码为 data URL、调用 OpenAI 并写出产物。 | 运行元数据和输出目录 |
| 说话人分离 | 调用 `gpt-4o-transcribe-diarize`，并设置 `response_format="diarized_json"` 和 `chunking_strategy="auto"`。 | 带说话人标签的片段 |
| 文字稿规范化 | 将 API 输出转换为带有稳定片段 ID 的一致 JSON 和 Markdown。 | `transcript_segments.json`、`speaker_labeled_transcript.md` |
| 会议智能 | 提取摘要、决策、行动项、风险、明确提出的问题、建议的跟进事项、引语以及跟进邮件，并附带结构化证据引用。 | `meeting_intelligence.json`、`meeting_brief.md` |
| 防护与审核关卡 | 对敏感字段脱敏、验证证据引用、可选地进行内容审核，并将有风险的输出转交审核。 | `guardrail_report.json` |

这里有意采用基于请求的方式。Realtime API 更适合实时语音交互体验、浏览器采集或电话流式传输。对于需要持久保存的通话后说话人分离，此模式使用 Transcriptions API，然后在带说话人标签的文字稿上运行结构化提取。


## 为什么能识别说话人的文字稿很重要

会议智能的第一个版本通常是“把文字稿发给模型并总结”。这在演示中可行，但在客户工作流中会失效，因为它丢失了“谁说了什么”。客户可能陈述了一项需求，销售方可能做出了一项承诺，而经理需要把两者的区别弄清楚。

能识别说话人的说话人分离为应用的其余部分提供了更好的结构：

- 行动项可以包含做出承诺的说话人。
- 风险可以引用客户的原话。
- 跟进邮件草稿可以避免把销售方的承诺归到客户名下。
- QA 审核人员可以按片段 ID 和时间戳抽查说话人归属。
- CRM 同步任务可以存储可机械验证的证据，而不是不透明的摘要。


## 安全与防护

会议智能应被视为一种敏感数据工作流，而不仅仅是转写或摘要任务。原始录音可能包含客户姓名、商业条款、支持细节、健康或财务信息以及内部策略。说话人参考片段也可能是敏感的，因为它们与某个人的声音绑定。一旦流水线将这些音频转化为结构化输出，这些输出就可能流入 CRM 记录、支持工单、客户计划、仪表盘或审核队列。

当输出可能影响业务流程时，安全与防护最为重要。一份销售通话摘要可能记录了定价或合同承诺。一次支持升级可能涉及影响生产的事故或客户凭证。一份招聘复盘可能包含对候选人的反馈。一场受监管行业的会议可能包含需要遵循保留、访问控制或脱敏策略的数据。对于这些工作流，最安全的模式是尽量减少原始音频的保留，在适当时对敏感内容脱敏，要求输出有证据支撑，并在写入下游之前将有风险或低置信度的输出交由人工审核。

其中包含的正则脱敏仅作示例：它只会遮盖基本的电子邮件和电话号码模式，并不是一个完整的 PII 或 DLP 系统。对于姓名、地址、账户标识、凭证、健康数据、财务数据或受监管的工作流，请使用经策略批准的 PII/DLP 检测器，并在写入下游之前保留人工审核关卡。

在结构化提取步骤中，本 notebook 在 Responses API 调用上设置了 `store=False`，这样生成的会议智能响应就不会作为应用状态被存储。`store=False` 是一个有用的请求级控制，但它并不等同于为组织或项目启用零数据保留（Zero Data Retention）。如果你的工作流需要更严格的数据保留保证，请查阅 OpenAI 的 [数据控制文档](https://platform.openai.com/docs/models/default-usage-policies-by-endpoint)，并为你的用例确认合适的保留配置。

| 风险 | 防护措施 |
| --- | --- |
| 录音或说话人参考被滥用 | 在录音、说话人分离或使用参考片段之前，要求获得同意和策略批准。将说话人参考视为近似生物特征的敏感数据。 |
| 原始音频保留过久 | 默认不保存原始转写响应。原始音频和参考片段只在需要的期间内保留。如需保留，请加密并限制访问。 |
| 文字稿中的提示词注入 | 将文字稿文本视为不可信的证据。将指令放在系统消息中，并要求模型只使用有文字稿支撑的事实。 |
| 没有依据的行动项或决策 | 使用严格的结构化输出，并要求证据引用指向真实的片段 ID 和引语。 |
| 生成笔记中的敏感内容 | 尽可能在摘要之前进行脱敏，然后在生成后对文字稿和简报进行检查。 |
| 有害或策略敏感的内容 | 可选地使用 `omni-moderation-latest` 对文字稿文本和生成的简报文本调用 Moderation API。内容审核用于检测有害内容，不能替代隐私审查。 |
| 不安全的下游写入 | 不要直接根据模型输出写入 CRM、工单或分析系统。对于中/高风险、缺失证据、审核标记或保留了原始响应的情况，在前面设置人工审核关卡。 |
| 悄无声息的质量漂移 | 记录模型版本、提示词版本、schema 版本、音频时长、脱敏状态、审核状态以及审核人员的决定。抽样通话用于评估。 |


## 前提条件

- Python 3.10 或更高版本。
- 用于真实音频运行的 OpenAI API 密钥，存放在 `OPENAI_API_KEY` 中。
- 用于真实音频运行的、受支持音频格式的会议录音。
- 上传的音频必须不超过 25 MB。支持的输入格式为 `mp3`、`mp4`、`mpeg`、`mpga`、`m4a`、`wav` 和 `webm`。
- 可选：最多四个简短的单说话人参考片段。语音转文本指南建议使用 2-10 秒的参考片段，并在通过 multipart 表单数据发送时编码为 data URL。

### 在本地运行 notebook

在 Cookbook 仓库的本地克隆中，创建虚拟环境，安装 Jupyter 和 OpenAI SDK，然后启动本 notebook：

```bash
git clone https://github.com/openai/openai-cookbook.git
cd openai-cookbook
python3 -m venv .venv
source .venv/bin/activate
python -m pip install jupyter "openai>=1.93.0"
export OPENAI_API_KEY="your-api-key"
jupyter notebook examples/audio/speaker_aware_meeting_intelligence/speaker_aware_meeting_intelligence.ipynb
```

下面的合成演示只使用 Python 标准库，不会调用 API。对于真实音频，你也可以在现有的 notebook 环境中安装 OpenAI SDK：

```python
%pip install "openai>=1.93.0"
```


## 核心说话人分离请求

核心 API 请求有意保持精简：

```python
client = OpenAI(timeout=30 * 60)

with open("meeting.wav", "rb") as audio_file:
    stream = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        response_format="diarized_json",
        chunking_strategy="auto",
        stream=True,
        extra_body={
            "known_speaker_names": ["Agent"],
            "known_speaker_references": [to_data_url(Path("agent_reference.wav"))],
        },
    )
    for event in stream:
        if event.type == "transcript.text.segment":
            print(event.speaker, event.text, event.start, event.end)
```

重要细节如下：

- 当你需要片段级的说话人元数据时，使用 `response_format="diarized_json"`。
- 对于超过 30 秒的音频，使用 `chunking_strategy="auto"`。
- 对于已完成的录音，如果希望在已定稿的说话人分离片段可用时就立即获取，使用 `stream=True`。
- Python SDK 默认的读取超时为 10 分钟；下面的辅助函数对较长的录音使用 30 分钟。
- 已知说话人的名称和参考片段要一起传入，并保持相同顺序。
- 参考片段要简短，且只包含单个说话人。


## 说话人分离与说话人识别

说话人分离回答的是“每个片段是哪个声音说的？”它在一段录音中区分不同的声音，但不会创建永久的身份档案，也不会记住某次通话中的 `speaker_0` 与之后另一次通话中的 `speaker_0` 是同一个人。即使没有参考片段，通用标签依然有用，因为它们保留了归属关系：流水线能够区分提出需求的说话人和做出承诺的说话人。

已知说话人参考为当前请求提供了一个可选的身份提示：

| 输入 | 结果 |
| --- | --- |
| 仅会议音频 | 模型会区分不同声音，通常使用 `speaker_0` 和 `speaker_1` 之类的通用标签。 |
| 会议音频加上一个带名称的参考片段 | 匹配的片段可以使用所提供的名称；未匹配的说话人可以保持通用标签。 |
| 之后的或历史的录音 | 再次传入参考片段。标签不会在不同录音之间自动沿用。 |

### 将参考片段与会议录音一起传入

会议录音和参考片段是同一个转写请求中的两个独立输入。不要把参考片段拼接到会议音频上。会议以 `file=...` 的形式上传；每个参考片段都编码为 data URL，并通过 `known_speaker_references` 发送，同时在 `known_speaker_names` 的相同位置提供名称。

下面的辅助函数封装了这种请求结构。例如，这里传入一段会议录音，外加一段内部销售代表讲话的独立短片段：

```python
meeting_audio = Path("customer_call.wav")
known_speakers = [
    ("Internal rep", Path("internal_rep_reference.wav")),
]

raw_transcription = transcribe_with_diarization(
    audio_file=meeting_audio,
    known_speakers=known_speakers,
)
```

请使用一段干净、已获同意、时长 2-10 秒、只有一个说话人且背景噪音极少的片段。对于经常出现的内部说话人，生产应用可以维护一个有访问控制的参考片段注册表，并在每次请求时附上相应的片段。对于历史录音，请对每段录音运行相同的流程；如果某段较早的、已获同意的通话中的片段干净且只有单个说话人，也可以将其用作参考片段。请将参考片段视为敏感数据，在有代表性的音频上评估匹配质量，并对高风险的下游写入保留人工审核。


In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
import os
import re
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

try:
    from IPython.display import JSON, Markdown, display
except ImportError:  # Makes this cell safe in non-notebook runners.
    JSON = None
    Markdown = None

    def display(value):
        print(value)


def show_markdown(text: str) -> None:
    if Markdown:
        display(Markdown(text))
    else:
        print(text)


def show_json(payload: Any, expanded: bool = False) -> None:
    if JSON:
        display(JSON(payload, expanded=expanded))
    else:
        print(json.dumps(payload, indent=2))


DEFAULT_TRANSCRIPTION_MODEL = "gpt-4o-transcribe-diarize"
DEFAULT_SUMMARY_MODEL = os.getenv("OPENAI_MEETING_INTELLIGENCE_MODEL", "gpt-4.1-mini")
DEFAULT_MODERATION_MODEL = "omni-moderation-latest"
SUPPORTED_REFERENCE_MIME_PREFIXES = ("audio/", "video/")
MAX_AUDIO_UPLOAD_BYTES = 25_000_000
DEFAULT_TRANSCRIPTION_TIMEOUT_SECONDS = 30 * 60

print("Notebook helpers loaded")


In [ ]:
@dataclass(frozen=True)
class Segment:
    segment_id: str
    speaker: str
    start: float
    end: float
    text: str


DEMO_SEGMENTS = [
    Segment(
        segment_id="seg_001",
        speaker="Solutions Engineer",
        start=0.0,
        end=9.2,
        text="Thanks for joining. I would like to understand where your support handoff breaks down today.",
    ),
    Segment(
        segment_id="seg_002",
        speaker="Customer",
        start=9.3,
        end=22.4,
        text="The biggest issue is that escalation notes are inconsistent. Managers spend Monday morning reconstructing what happened from call recordings.",
    ),
    Segment(
        segment_id="seg_003",
        speaker="Solutions Engineer",
        start=22.5,
        end=38.1,
        text="So the priority is reliable call summaries, who committed to what, and enough evidence that the team trusts the handoff.",
    ),
    Segment(
        segment_id="seg_004",
        speaker="Customer",
        start=38.2,
        end=55.0,
        text="Exactly. We also need risks called out, especially compliance-sensitive promises, and we need to push action items into our CRM.",
    ),
    Segment(
        segment_id="seg_005",
        speaker="Solutions Engineer",
        start=55.1,
        end=70.3,
        text="I will send a prototype that includes speaker-aware transcripts, action items with evidence, and a redaction pass before CRM sync.",
    ),
]


PII_PATTERNS = [
    (re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"), "[email]"),
    (re.compile(r"\b(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}\b"), "[phone]"),
]

print(f"Loaded {len(DEMO_SEGMENTS)} synthetic transcript segments")


## 第 1 步：定义结构化输出 schema

会议智能的结果经常被送入记录系统。使用严格的结构化输出，可以让下游代码获得稳定的数据结构，不受支持的字段会被拒绝，而不是被悄悄接受。这个 schema 还要求提供结构化的 `evidence_refs`：每个提取出的条目都必须引用文字稿中的一个 `segment_id` 以及该片段中的一句引语，这样防护机制就能机械地验证其依据。


In [ ]:
EVIDENCE_REF_SCHEMA: dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "segment_id": {"type": "string"},
        "quote": {"type": "string"},
    },
    "required": ["segment_id", "quote"],
}

EVIDENCE_REFS_SCHEMA: dict[str, Any] = {
    "type": "array",
    "items": EVIDENCE_REF_SCHEMA,
}

NULLABLE_STRING_SCHEMA: dict[str, Any] = {"type": ["string", "null"]}


MEETING_INTELLIGENCE_SCHEMA: dict[str, Any] = {
    "name": "meeting_intelligence",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "summary": {"type": "string"},
            "participants": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "speaker": {"type": "string"},
                        "inferred_role": NULLABLE_STRING_SCHEMA,
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["speaker", "inferred_role", "evidence_refs"],
                },
            },
            "customer_context": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "fact": {"type": "string"},
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["fact", "evidence_refs"],
                },
            },
            "decisions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "decision": {"type": "string"},
                        "speaker_or_group": NULLABLE_STRING_SCHEMA,
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["decision", "speaker_or_group", "evidence_refs"],
                },
            },
            "action_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "owner_speaker": NULLABLE_STRING_SCHEMA,
                        "task": {"type": "string"},
                        "due_date_or_trigger": NULLABLE_STRING_SCHEMA,
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["owner_speaker", "task", "due_date_or_trigger", "evidence_refs"],
                },
            },
            "risks": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "risk": {"type": "string"},
                        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                        "mitigation": {"type": "string"},
                    },
                    "required": ["risk", "severity", "evidence_refs", "mitigation"],
                },
            },
            "explicit_questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "question": {"type": "string"},
                        "asked_by_speaker": {"type": "string"},
                        "directed_to_speaker": NULLABLE_STRING_SCHEMA,
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["question", "asked_by_speaker", "directed_to_speaker", "evidence_refs"],
                },
            },
            "suggested_follow_ups": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "question": {"type": "string"},
                        "rationale": {"type": "string"},
                        "evidence_refs": EVIDENCE_REFS_SCHEMA,
                    },
                    "required": ["question", "rationale", "evidence_refs"],
                },
            },
            "notable_quotes": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "speaker": {"type": "string"},
                        "quote": {"type": "string"},
                        "timestamp": {"type": "string"},
                        "segment_id": {"type": "string"},
                    },
                    "required": ["speaker", "quote", "timestamp", "segment_id"],
                },
            },
            "follow_up_email": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "subject": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["subject", "body"],
            },
        },
        "required": [
            "summary",
            "participants",
            "customer_context",
            "decisions",
            "action_items",
            "risks",
            "explicit_questions",
            "suggested_follow_ups",
            "notable_quotes",
            "follow_up_email",
        ],
    },
}

print("Structured output schema ready")


## 第 2 步：构建音频和文字稿辅助函数

已知说话人参考是可选的。没有参考时，说话人分离仍然可以区分说话人，但标签可能是通用的，例如 `speaker_0` 或 `speaker_1`。有了参考，API 就可以将片段映射到你提供的名称上。

请使用简短、干净、只有一个说话人且背景噪音极少的参考片段。只有在获得同意且有明确业务需求时才保留参考片段。


In [ ]:
def to_data_url(path) -> str:
    path = Path(path)
    mime_type, _ = mimetypes.guess_type(path)
    if mime_type is None:
        mime_type = "audio/wav"
    elif not mime_type.startswith(SUPPORTED_REFERENCE_MIME_PREFIXES):
        raise ValueError(f"Reference clip must be an audio or video file, got {mime_type}: {path}")
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


def transcribe_with_diarization(
    audio_file: Path,
    known_speakers: list[tuple[str, Path]],
    model: str = DEFAULT_TRANSCRIPTION_MODEL,
    request_timeout_seconds: float = DEFAULT_TRANSCRIPTION_TIMEOUT_SECONDS,
    stream_transcription: bool = True,
) -> Any:
    if not audio_file.is_file():
        raise FileNotFoundError(f"Audio file does not exist or is not a regular file: {audio_file}")

    if request_timeout_seconds <= 0:
        raise ValueError("request_timeout_seconds must be positive.")

    audio_size_bytes = audio_file.stat().st_size
    if audio_size_bytes > MAX_AUDIO_UPLOAD_BYTES:
        raise ValueError(
            f"Audio file is {audio_size_bytes:,} bytes; "
            "the Audio Transcriptions API accepts uploads up to 25 MB. "
            "Compress or split the recording before retrying."
        )

    from openai import OpenAI

    client = OpenAI(timeout=request_timeout_seconds)
    params: dict[str, Any] = {
        "model": model,
        "response_format": "diarized_json",
        "chunking_strategy": "auto",
        "stream": stream_transcription,
    }

    if known_speakers:
        if len(known_speakers) > 4:
            raise ValueError("gpt-4o-transcribe-diarize accepts up to 4 known speaker references.")
        params["extra_body"] = {
            "known_speaker_names": [name for name, _ in known_speakers],
            "known_speaker_references": [to_data_url(path) for _, path in known_speakers],
        }

    with audio_file.open("rb") as audio:
        response = client.audio.transcriptions.create(file=audio, **params)
        if stream_transcription:
            return collect_streamed_transcription(response)
        return response


def to_plain(value: Any) -> Any:
    if hasattr(value, "model_dump"):
        return value.model_dump()
    if isinstance(value, dict):
        return {key: to_plain(inner) for key, inner in value.items()}
    if isinstance(value, list):
        return [to_plain(item) for item in value]
    return value


def collect_streamed_transcription(events: Any) -> dict[str, Any]:
    segments: list[dict[str, Any]] = []
    full_text = ""
    usage: Any = None

    for event in events:
        data = to_plain(event)
        if not isinstance(data, dict):
            continue

        event_type = data.get("type")
        if event_type == "transcript.text.segment":
            segments.append(data)
        elif event_type == "transcript.text.done":
            full_text = str(data.get("text") or "")
            usage = data.get("usage")

    if not segments:
        raise ValueError("No diarized transcript segments were emitted by the transcription stream.")
    return {"segments": segments, "text": full_text, "usage": usage}


def normalize_segments(transcription: Any) -> list[Segment]:
    data = to_plain(transcription)
    raw_segments = data.get("segments", []) if isinstance(data, dict) else []
    segments: list[Segment] = []

    for index, item in enumerate(raw_segments):
        if hasattr(item, "model_dump"):
            item = item.model_dump()
        if not isinstance(item, dict):
            continue

        text = str(item.get("text", "")).strip()
        if not text:
            continue

        segment_id = str(item.get("segment_id") or item.get("id") or f"seg_{len(segments) + 1:03d}")
        segments.append(
            Segment(
                segment_id=segment_id,
                speaker=str(item.get("speaker") or f"Speaker {index + 1}"),
                start=float(item.get("start") or 0.0),
                end=float(item.get("end") or 0.0),
                text=text,
            )
        )

    if not segments and isinstance(data, dict) and data.get("text"):
        segments.append(Segment(segment_id="seg_001", speaker="Speaker 1", start=0.0, end=0.0, text=str(data["text"])))

    if not segments:
        raise ValueError("No transcript segments were found in the transcription response.")
    return segments

print("Audio and transcript helpers ready")


## 第 3 步：规范化文字稿

规范化后的文字稿是音频处理与会议智能之间的契约。它让你无需重新转写音频即可重新运行摘要，便于检查归属质量，并缩短原始音频的保留时间。

每个片段都会获得一个稳定的 `segment_id`，例如 `seg_005`。之后，模型必须在 `evidence_refs` 中引用这些 ID，而防护步骤会验证每条被引用的引语确实出现在所引用的片段中。

下面的正则脱敏辅助函数仅作示例：它只会遮盖基本的电子邮件和电话号码模式，并不是一个完整的 PII 或 DLP 系统；对于敏感或受监管的工作流，请使用经策略批准的检测器并进行人工审核。


In [ ]:
def redact_text(text: str) -> str:
    redacted = text
    for pattern, replacement in PII_PATTERNS:
        redacted = pattern.sub(replacement, redacted)
    return redacted


def redact_segments(segments: list[Segment]) -> list[Segment]:
    return [
        Segment(
            segment_id=segment.segment_id,
            speaker=segment.speaker,
            start=segment.start,
            end=segment.end,
            text=redact_text(segment.text),
        )
        for segment in segments
    ]


def pii_matches(text: str) -> list[str]:
    matches: list[str] = []
    for pattern, replacement in PII_PATTERNS:
        if pattern.search(text):
            matches.append(replacement.strip("[]"))
    return sorted(set(matches))


def format_timestamp(seconds: float) -> str:
    total_ms = max(0, int(round(seconds * 1000)))
    minutes, remainder_ms = divmod(total_ms, 60_000)
    secs, millis = divmod(remainder_ms, 1000)
    return f"{minutes:02d}:{secs:02d}.{millis:03d}"


def transcript_as_markdown(segments: list[Segment]) -> str:
    lines = ["# Speaker-Labeled Transcript", ""]
    for segment in segments:
        start = format_timestamp(segment.start)
        end = format_timestamp(segment.end)
        lines.append(f"**{segment.segment_id} | {segment.speaker} [{start}-{end}]**: {segment.text}")
        lines.append("")
    return "\n".join(lines).rstrip() + "\n"


def transcript_for_model(segments: list[Segment]) -> str:
    return "\n".join(
        f"{segment.segment_id} | {segment.speaker} | {format_timestamp(segment.start)}-{format_timestamp(segment.end)} | {segment.text}"
        for segment in segments
    )


show_markdown(transcript_as_markdown(DEMO_SEGMENTS))


## 第 4 步：提取结构化会议智能

模型会收到一份带说话人标签的文字稿，并且只能将该文字稿作为证据。最安全的默认做法是输出空数组，而不是生成看似合理却缺乏依据的 CRM 笔记。该 schema 还使用了必填但可为空的字段，例如 `due_date_or_trigger`、`inferred_role` 和 `directed_to_speaker`，这样未知值会保持为 `null`，而不会被猜测填充。

对于每一个提取出的事实、行动项、风险、问题或建议，模型都会返回带有 `segment_id` 和引语的 `evidence_refs`。这为审核人员提供了一条可读的来源线索，也为代码提供了可供验证的具体内容。


In [ ]:
def response_output_text_or_raise(response: Any) -> str:
    data = to_plain(response)
    status = data.get("status") if isinstance(data, dict) else getattr(response, "status", None)
    if status and status != "completed":
        details = data.get("incomplete_details") if isinstance(data, dict) else getattr(response, "incomplete_details", None)
        raise RuntimeError(f"Responses API returned status={status!r}; incomplete_details={details!r}")

    refusals: list[str] = []
    if isinstance(data, dict):
        for item in data.get("output", []):
            if not isinstance(item, dict):
                continue
            for content in item.get("content", []):
                if not isinstance(content, dict):
                    continue
                if content.get("type") == "refusal" or content.get("refusal"):
                    refusals.append(str(content.get("refusal") or content.get("text") or content))
    if refusals:
        raise RuntimeError(f"Responses API returned a refusal: {refusals[0]}")

    content = getattr(response, "output_text", None)
    if content is None and isinstance(data, dict):
        content = data.get("output_text")
    content = str(content or "").strip()
    if not content:
        raise RuntimeError("The model returned an empty response.")
    return content


def parse_meeting_intelligence_json(content: str) -> dict[str, Any]:
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Responses API returned invalid JSON: {exc}") from exc
    if not isinstance(parsed, dict):
        raise RuntimeError("Responses API returned JSON, but the top-level value was not an object.")
    return parsed


def generate_meeting_intelligence(segments: list[Segment], model: str = DEFAULT_SUMMARY_MODEL) -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI()
    transcript = transcript_for_model(segments)

    completion = client.responses.create(
        model=model,
        temperature=0,
        store=False,
        input=[
            {
                "role": "system",
                "content": (
                    "You create meeting intelligence from speaker-labeled transcripts. "
                    "Use only the transcript as evidence. Do not invent names, dates, decisions, "
                    "commitments, or implementation details. If evidence is missing, leave the relevant array empty. "
                    "Use null for unknown roles, owners, due dates or triggers, directed-to speakers, or decision owners. "
                    "Put single-speaker commitments in action_items, not decisions. "
                    "Only include decisions when the transcript shows an explicit decision or agreement. "
                    "Put only questions actually asked in explicit_questions. "
                    "Put inferred next questions in suggested_follow_ups with rationale and evidence_refs. "
                    "Every extracted item must include evidence_refs with segment_id values copied from the transcript "
                    "and quote text copied from that same segment. Do not fabricate segment IDs or quotes. "
                    "Use empty arrays instead of unsupported items. "
                    "If the follow-up email signer is unknown, end with [Your name]."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Extract a customer-safe meeting brief from this transcript. "
                    "Transcript rows use: segment_id | speaker | timestamp range | text.\n\n"
                    f"{transcript}"
                ),
            },
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": MEETING_INTELLIGENCE_SCHEMA["name"],
                "strict": MEETING_INTELLIGENCE_SCHEMA["strict"],
                "schema": MEETING_INTELLIGENCE_SCHEMA["schema"],
            }
        },
    )

    content = response_output_text_or_raise(completion)
    return parse_meeting_intelligence_json(content)


def demo_meeting_intelligence() -> dict[str, Any]:
    return {
        "summary": (
            "The customer needs a dependable post-call handoff process. Their main pain point is "
            "inconsistent escalation notes, which forces managers to reconstruct calls manually. "
            "The proposed path is a speaker-aware transcript, evidence-backed action items, risk "
            "detection, redaction, and CRM sync."
        ),
        "participants": [
            {
                "speaker": "Solutions Engineer",
                "inferred_role": "OpenAI technical seller or solution owner",
                "evidence_refs": [
                    {
                        "segment_id": "seg_001",
                        "quote": "I would like to understand where your support handoff breaks down today.",
                    },
                    {
                        "segment_id": "seg_005",
                        "quote": "I will send a prototype that includes speaker-aware transcripts, action items with evidence, and a redaction pass before CRM sync.",
                    },
                ],
            },
            {
                "speaker": "Customer",
                "inferred_role": "Customer stakeholder for support operations",
                "evidence_refs": [
                    {
                        "segment_id": "seg_002",
                        "quote": "The biggest issue is that escalation notes are inconsistent.",
                    },
                    {
                        "segment_id": "seg_004",
                        "quote": "we need to push action items into our CRM.",
                    },
                ],
            },
        ],
        "customer_context": [
            {
                "fact": "Escalation notes are inconsistent today.",
                "evidence_refs": [
                    {
                        "segment_id": "seg_002",
                        "quote": "The biggest issue is that escalation notes are inconsistent.",
                    }
                ],
            },
            {
                "fact": "Managers spend time reconstructing calls from recordings.",
                "evidence_refs": [
                    {
                        "segment_id": "seg_002",
                        "quote": "Managers spend Monday morning reconstructing what happened from call recordings.",
                    }
                ],
            },
            {
                "fact": "The customer wants action items pushed into their CRM.",
                "evidence_refs": [{"segment_id": "seg_004", "quote": "we need to push action items into our CRM."}],
            },
        ],
        "decisions": [],
        "action_items": [
            {
                "owner_speaker": "Solutions Engineer",
                "task": "Send a prototype that includes speaker-aware transcripts, action items with evidence, and a redaction pass before CRM sync.",
                "due_date_or_trigger": None,
                "evidence_refs": [
                    {
                        "segment_id": "seg_005",
                        "quote": "I will send a prototype that includes speaker-aware transcripts, action items with evidence, and a redaction pass before CRM sync.",
                    }
                ],
            }
        ],
        "risks": [
            {
                "risk": "Compliance-sensitive promises need to be identified in meeting notes.",
                "severity": "medium",
                "evidence_refs": [
                    {
                        "segment_id": "seg_004",
                        "quote": "We also need risks called out, especially compliance-sensitive promises",
                    }
                ],
                "mitigation": "Route compliance-sensitive risks to human review before CRM sync.",
            }
        ],
        "explicit_questions": [
            {
                "question": "Where does your support handoff break down today?",
                "asked_by_speaker": "Solutions Engineer",
                "directed_to_speaker": "Customer",
                "evidence_refs": [
                    {
                        "segment_id": "seg_001",
                        "quote": "I would like to understand where your support handoff breaks down today.",
                    }
                ],
            }
        ],
        "suggested_follow_ups": [
            {
                "question": "Which CRM object and fields should receive action items?",
                "rationale": "The customer asked to push action items into their CRM but did not specify the target schema or workflow.",
                "evidence_refs": [{"segment_id": "seg_004", "quote": "we need to push action items into our CRM."}],
            }
        ],
        "notable_quotes": [
            {
                "speaker": "Customer",
                "quote": "Managers spend Monday morning reconstructing what happened from call recordings.",
                "timestamp": "00:09.300",
                "segment_id": "seg_002",
            }
        ],
        "follow_up_email": {
            "subject": "Prototype for speaker-aware meeting handoffs",
            "body": (
                "Hi,\n\nThanks for the conversation. I heard that inconsistent escalation notes, "
                "evidence-backed action items, compliance-sensitive risk detection, and CRM sync "
                "are the core requirements. I will send a prototype with speaker-aware transcripts, "
                "action items with evidence, and a redaction pass before CRM sync.\n\nBest,\n[Your name]"
            ),
        },
    }


show_json(demo_meeting_intelligence(), expanded=False)


## 第 5 步：渲染可供审核的会议简报

审核产物将说话人、片段 ID、时间戳和引语证据与决策、风险和行动项放在一起，以便人工在任何内容写入下游之前进行抽查。


In [ ]:
def clean_markdown_cell(value: Any) -> str:
    if value is None:
        return "_Not specified._"
    return str(value).replace("|", "\\|").replace("\n", "<br>")


def render_evidence_refs(refs: Any) -> str:
    if not isinstance(refs, list) or not refs:
        return "_No evidence refs._"

    rendered = []
    for ref in refs:
        if not isinstance(ref, dict):
            continue
        segment_id = clean_markdown_cell(ref.get("segment_id", ""))
        quote = clean_markdown_cell(ref.get("quote", ""))
        rendered.append(f"`{segment_id}`: {quote}")
    return "<br>".join(rendered) if rendered else "_No evidence refs._"


def markdown_table(rows: list[dict[str, Any]], columns: list[tuple[str, Any]]) -> str:
    if not rows:
        return "_None identified._"

    header = "| " + " | ".join(title for title, _ in columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            value = key(row) if callable(key) else row.get(key, "")
            values.append(clean_markdown_cell(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, divider, *body])


def render_meeting_brief(intelligence: dict[str, Any]) -> str:
    follow_up = intelligence.get("follow_up_email", {})
    evidence_column = ("Evidence", lambda row: render_evidence_refs(row.get("evidence_refs", [])))
    lines = [
        "# Meeting Brief",
        "",
        "## Summary",
        "",
        str(intelligence.get("summary", "")).strip() or "_No summary generated._",
        "",
        "## Participants",
        "",
        markdown_table(
            intelligence.get("participants", []),
            [("Speaker", "speaker"), ("Inferred role", "inferred_role"), evidence_column],
        ),
        "",
        "## Customer Context",
        "",
        markdown_table(intelligence.get("customer_context", []), [("Fact", "fact"), evidence_column]),
        "",
        "## Decisions",
        "",
        markdown_table(
            intelligence.get("decisions", []),
            [("Decision", "decision"), ("Owner", "speaker_or_group"), evidence_column],
        ),
        "",
        "## Action Items",
        "",
        markdown_table(
            intelligence.get("action_items", []),
            [
                ("Owner", "owner_speaker"),
                ("Task", "task"),
                ("Due date or trigger", "due_date_or_trigger"),
                evidence_column,
            ],
        ),
        "",
        "## Risks",
        "",
        markdown_table(
            intelligence.get("risks", []),
            [("Risk", "risk"), ("Severity", "severity"), evidence_column, ("Mitigation", "mitigation")],
        ),
        "",
        "## Explicit Questions",
        "",
        markdown_table(
            intelligence.get("explicit_questions", []),
            [
                ("Question", "question"),
                ("Asked by", "asked_by_speaker"),
                ("Directed to", "directed_to_speaker"),
                evidence_column,
            ],
        ),
        "",
        "## Suggested Follow-ups",
        "",
        markdown_table(
            intelligence.get("suggested_follow_ups", []),
            [("Question", "question"), ("Rationale", "rationale"), evidence_column],
        ),
        "",
        "## Notable Quotes",
        "",
        markdown_table(
            intelligence.get("notable_quotes", []),
            [("Speaker", "speaker"), ("Quote", "quote"), ("Timestamp", "timestamp"), ("Segment ID", "segment_id")],
        ),
        "",
        "## Follow-up Email Draft",
        "",
        f"**Subject:** {follow_up.get('subject', '')}",
        "",
        str(follow_up.get("body", "")).strip(),
        "",
    ]
    return "\n".join(lines).rstrip() + "\n"


demo_brief = render_meeting_brief(demo_meeting_intelligence())
show_markdown(demo_brief)


## 第 6 步：添加防护并写出产物

示例会写出一份 `guardrail_report.json`，包含以下本地检查：

- 规范化后的文字稿片段；
- 基本的电子邮件和电话号码 PII 模式；
- 指向真实片段 ID 和匹配引语的证据引用；
- 中/高风险输出；
- 可选的内容审核标记；
- 原始转写响应的存储情况。


In [ ]:
def summarize_moderation_response(response: Any) -> dict[str, Any]:
    data = to_plain(response)
    summaries: list[dict[str, Any]] = []

    for result in data.get("results", []) if isinstance(data, dict) else []:
        categories = result.get("categories", {}) if isinstance(result, dict) else {}
        category_scores = result.get("category_scores", {}) if isinstance(result, dict) else {}
        flagged_categories = sorted(key for key, value in categories.items() if bool(value))
        top_scores = dict(sorted(category_scores.items(), key=lambda item: float(item[1] or 0.0), reverse=True)[:5])
        summaries.append(
            {
                "flagged": bool(result.get("flagged")) if isinstance(result, dict) else False,
                "flagged_categories": flagged_categories,
                "top_category_scores": top_scores,
            }
        )

    return {
        "id": data.get("id") if isinstance(data, dict) else None,
        "model": data.get("model") if isinstance(data, dict) else DEFAULT_MODERATION_MODEL,
        "flagged": any(item["flagged"] for item in summaries),
        "results": summaries,
    }


def moderate_text(text: str, model: str = DEFAULT_MODERATION_MODEL) -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI()
    response = client.moderations.create(model=model, input=text)
    return summarize_moderation_response(response)


def iter_evidence_refs(value: Any, path: str = "$") -> list[tuple[str, Any]]:
    found: list[tuple[str, Any]] = []
    if isinstance(value, dict):
        for key, inner in value.items():
            next_path = f"{path}.{key}"
            if key == "evidence_refs":
                found.append((next_path, inner))
            else:
                found.extend(iter_evidence_refs(inner, next_path))
    elif isinstance(value, list):
        for index, item in enumerate(value):
            found.extend(iter_evidence_refs(item, f"{path}[{index}]"))
    return found


def normalize_for_quote_match(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().casefold()


def validate_evidence_refs(intelligence: dict[str, Any], segments: list[Segment]) -> list[dict[str, Any]]:
    segment_by_id = {segment.segment_id: segment for segment in segments}
    problems: list[dict[str, Any]] = []

    for path, refs in iter_evidence_refs(intelligence):
        if not isinstance(refs, list) or not refs:
            problems.append({"path": path, "issue": "missing_or_empty_evidence_refs"})
            continue

        for index, ref in enumerate(refs):
            ref_path = f"{path}[{index}]"
            if not isinstance(ref, dict):
                problems.append({"path": ref_path, "issue": "evidence_ref_is_not_an_object"})
                continue

            segment_id = str(ref.get("segment_id", "")).strip()
            quote = str(ref.get("quote", "")).strip()
            if not segment_id or not quote:
                problems.append({"path": ref_path, "issue": "missing_segment_id_or_quote", "segment_id": segment_id})
                continue

            segment = segment_by_id.get(segment_id)
            if segment is None:
                problems.append({"path": ref_path, "issue": "unknown_segment_id", "segment_id": segment_id})
                continue

            if normalize_for_quote_match(quote) not in normalize_for_quote_match(segment.text):
                problems.append(
                    {
                        "path": ref_path,
                        "issue": "quote_not_found_in_segment",
                        "segment_id": segment_id,
                        "quote": quote,
                    }
                )

    for index, quote in enumerate(intelligence.get("notable_quotes", [])):
        if not isinstance(quote, dict):
            continue
        segment_id = str(quote.get("segment_id", "")).strip()
        quote_text = str(quote.get("quote", "")).strip()
        segment = segment_by_id.get(segment_id)
        if segment is None:
            problems.append({"path": f"$.notable_quotes[{index}]", "issue": "unknown_segment_id", "segment_id": segment_id})
        elif normalize_for_quote_match(quote_text) not in normalize_for_quote_match(segment.text):
            problems.append(
                {
                    "path": f"$.notable_quotes[{index}]",
                    "issue": "quote_not_found_in_segment",
                    "segment_id": segment_id,
                    "quote": quote_text,
                }
            )

    return problems


def add_guardrail_check(checks: list[dict[str, Any]], name: str, status: str, detail: str, evidence = None) -> None:
    check: dict[str, Any] = {"name": name, "status": status, "detail": detail}
    if evidence is not None:
        check["evidence"] = evidence
    checks.append(check)


def build_guardrail_report(
    segments: list[Segment],
    intelligence: dict[str, Any],
    meeting_brief: str,
    redaction_enabled: bool,
    raw_saved: bool,
    moderation_results: dict[str, Any],
) -> dict[str, Any]:
    checks: list[dict[str, Any]] = []
    transcript_text = transcript_for_model(segments)

    add_guardrail_check(
        checks,
        "transcript_segments_present",
        "pass" if segments else "fail",
        f"Found {len(segments)} normalized transcript segments.",
    )

    pii_found = pii_matches(transcript_text + "\n" + meeting_brief)
    pii_detail = (
        "Basic PII patterns remain after redaction."
        if redaction_enabled
        else "Basic PII patterns were detected; run with redaction or review before storage."
    )
    add_guardrail_check(
        checks,
        "basic_pii_scan",
        "review" if pii_found else "pass",
        pii_detail if pii_found else "No basic email or phone patterns detected.",
        {"matches": pii_found, "redaction_enabled": redaction_enabled},
    )

    evidence_ref_problems = validate_evidence_refs(intelligence, segments)
    add_guardrail_check(
        checks,
        "evidence_refs",
        "review" if evidence_ref_problems else "pass",
        (
            "Some evidence references are missing, cite unknown segments, or quote text that is not present in the cited segment."
            if evidence_ref_problems
            else "All evidence references point to real segments with matching quote text."
        ),
        {"problem_count": len(evidence_ref_problems), "examples": evidence_ref_problems[:5]},
    )

    risks = intelligence.get("risks", [])
    review_risks = [risk for risk in risks if str(risk.get("severity", "")).lower() in {"medium", "high"}]
    severity_counts = {
        "low": sum(1 for risk in risks if str(risk.get("severity", "")).lower() == "low"),
        "medium": sum(1 for risk in risks if str(risk.get("severity", "")).lower() == "medium"),
        "high": sum(1 for risk in risks if str(risk.get("severity", "")).lower() == "high"),
    }
    add_guardrail_check(
        checks,
        "risk_outputs",
        "review" if review_risks else "pass",
        "Medium or high risks should be reviewed before downstream writes." if review_risks else "No medium or high risks identified.",
        {"severity_counts": severity_counts},
    )

    moderation_flagged = [name for name, result in moderation_results.items() if isinstance(result, dict) and result.get("flagged")]
    if moderation_results:
        add_guardrail_check(
            checks,
            "moderation",
            "review" if moderation_flagged else "pass",
            "Moderation flagged content that should be reviewed." if moderation_flagged else "Moderation did not flag transcript or brief content.",
            {"flagged_artifacts": moderation_flagged},
        )
    else:
        add_guardrail_check(checks, "moderation", "not_run", "Moderation was not requested. Use moderation for content safety classification.")

    add_guardrail_check(
        checks,
        "raw_response_storage",
        "review" if raw_saved else "pass",
        "Raw transcription response was saved; confirm retention and access controls." if raw_saved else "Raw transcription response was not saved.",
    )

    status = "review_required" if any(check["status"] in {"review", "fail"} for check in checks) else "pass"
    if any(check["status"] == "fail" for check in checks):
        status = "fail"

    return {
        "status": status,
        "recommended_next_step": "Send artifacts to human review before downstream writes." if status != "pass" else "Artifacts passed local guardrail checks.",
        "checks": checks,
        "moderation": moderation_results,
    }

print("Guardrail helpers ready")


In [ ]:
def write_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def write_artifacts(
    output_dir: Path,
    segments: list[Segment],
    intelligence: dict[str, Any],
    guardrail_report: dict[str, Any],
    raw_payload = None,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    write_json(output_dir / "transcript_segments.json", [asdict(segment) for segment in segments])
    (output_dir / "speaker_labeled_transcript.md").write_text(transcript_as_markdown(segments), encoding="utf-8")
    write_json(output_dir / "meeting_intelligence.json", intelligence)
    (output_dir / "meeting_brief.md").write_text(render_meeting_brief(intelligence), encoding="utf-8")
    write_json(output_dir / "guardrail_report.json", guardrail_report)
    if raw_payload is not None:
        write_json(output_dir / "raw_transcription_response.json", to_plain(raw_payload))


def run_pipeline_from_segments(
    segments: list[Segment],
    output_dir: Path,
    intelligence = None,
    redaction_enabled: bool = False,
    moderation_results = None,
    raw_saved: bool = False,
    raw_payload = None,
) -> dict[str, Any]:
    if redaction_enabled:
        segments = redact_segments(segments)
    if intelligence is None:
        intelligence = generate_meeting_intelligence(segments)
    meeting_brief = render_meeting_brief(intelligence)
    guardrail_report = build_guardrail_report(
        segments=segments,
        intelligence=intelligence,
        meeting_brief=meeting_brief,
        redaction_enabled=redaction_enabled,
        raw_saved=raw_saved,
        moderation_results=moderation_results or {},
    )
    write_artifacts(output_dir, segments, intelligence, guardrail_report, raw_payload=raw_payload if raw_saved else None)
    return {
        "segments": segments,
        "intelligence": intelligence,
        "meeting_brief": meeting_brief,
        "guardrail_report": guardrail_report,
        "output_dir": output_dir,
    }

print("Artifact helpers ready")


## 第 7 步：运行确定性的演示 fixture

本节是一个确定性的、无网络的演示，而不是模型质量评估。它使用一份固定的合成说话人分离文字稿和一个固定的预期会议智能对象，让审核人员无需 API 密钥即可运行本 notebook。

### 这个 fixture 检查什么

该 fixture 演练的是与真实音频相同的产物和防护路径：文字稿渲染、JSON 写出、Markdown 简报渲染、PII 脱敏辅助函数、证据引用校验、可空字段以及审核路由。

### 这个 fixture 不检查什么

它不衡量新会议上的转写质量、说话人分离准确率或模型提取质量。下面的评估部分为这一层添加了确定性评分以及一个可选的 LLM 作为评判者（LLM-as-judge）模式。


In [ ]:
output_dir = Path(tempfile.mkdtemp(prefix="meeting-intelligence-demo-"))
demo_run = run_pipeline_from_segments(
    segments=DEMO_SEGMENTS,
    output_dir=output_dir,
    intelligence=demo_meeting_intelligence(),
)

print(f"Wrote meeting intelligence artifacts to {output_dir}")
for artifact_name in [
    "transcript_segments.json",
    "speaker_labeled_transcript.md",
    "meeting_intelligence.json",
    "meeting_brief.md",
    "guardrail_report.json",
]:
    artifact_path = output_dir / artifact_name
    print(f"- {artifact_path} ({artifact_path.stat().st_size} bytes)")


In [ ]:
segments = json.loads((output_dir / "transcript_segments.json").read_text())
segments[:2]


In [ ]:
show_markdown((output_dir / "speaker_labeled_transcript.md").read_text())


In [ ]:
meeting_intelligence_json = json.loads((output_dir / "meeting_intelligence.json").read_text())
show_json(meeting_intelligence_json, expanded=False)


In [ ]:
show_markdown((output_dir / "meeting_brief.md").read_text())


In [ ]:
guardrail_report = json.loads((output_dir / "guardrail_report.json").read_text())
show_json(guardrail_report, expanded=True)


## 第 8 步：使用真实音频运行

下一个单元格需要主动开启。设置 `RUN_REAL_AUDIO = True`，提供本地音频路径，并确保已设置 `OPENAI_API_KEY`。

Transcriptions API 接受不超过 25 MB 的文件。`chunking_strategy="auto"` 会对有效的上传进行分段，但不会拆分超大文件。对于更大的会议，请在转写之前将录音压缩为受支持的较低码率格式，或将其拆分为大小受限的多个文件，然后在合并结果时保留或偏移时间戳。
较长的录音还可能超过 Python SDK 默认的读取超时。辅助函数默认以流式方式获取已定稿的说话人分离片段，并保留 30 分钟的超时作为兜底。只有在你确实需要一次性获取非流式响应时，才设置 `stream_transcription=False`；当单个请求在运维上不够可靠时，请拆分特别长的录音。

### 读者如何提供自己的文件

本 cookbook 以 notebook 为主；发布的产物中没有单独的 `.py` 命令。请将会议录音以及任何可选的参考片段放在 notebook 内核可以读取的位置。在本地 Jupyter 中，可以是 notebook 旁边的一个文件夹，也可以是磁盘上的绝对路径。在托管的 notebook 中，请先将文件上传到 notebook 会话中。

例如，读者可能有：

```text
audio/
  customer_call.mp3
  internal_rep_reference.wav
```

然后将配置变量指向这些文件：

```python
AUDIO_FILE = Path("audio/customer_call.mp3")
KNOWN_SPEAKERS = {
    "Internal rep": Path("audio/internal_rep_reference.wav"),
}
```

`AUDIO_FILE` 是原始会议录音。`KNOWN_SPEAKERS` 将你希望在输出中使用的标签映射到一段独立的 2-10 秒参考片段；辅助函数会将该片段随请求一起发送，而不是将其追加到会议音频后面。在生产应用中，同一个辅助函数可以接收由上传内容创建、或从对象存储下载的临时文件。

对于第一次生产风格的运行，请保持设置简单：

- 使用一个会议音频文件。
- 对较长的录音使用 `chunking_strategy="auto"`。
- 只有在获得同意且有明确业务需求时，才添加已知说话人参考。
- 在存储之前进行脱敏。
- 当有害内容分类是你审核策略的一部分时，运行内容审核。


In [ ]:
RUN_REAL_AUDIO = False
AUDIO_FILE = Path("/path/to/meeting.wav")
# Keep each reference clip separate from AUDIO_FILE. Use a clean, consented 2-10 second sample of one speaker.
KNOWN_SPEAKERS = {
    # "Internal rep": Path("/path/to/internal_rep_reference.wav"),
    # "Customer": Path("/path/to/customer_reference.wav"),
}
# Demonstration only: masks basic email and phone patterns, not a complete PII/DLP solution.
REDACT_REAL_AUDIO = True
MODERATE_REAL_AUDIO = False
SAVE_RAW_RESPONSE = False
REAL_OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="meeting-intelligence-real-"))

if RUN_REAL_AUDIO:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running on real audio.")
    if not AUDIO_FILE.is_file():
        raise FileNotFoundError(AUDIO_FILE)

    known_speakers = [(speaker_name, reference_path) for speaker_name, reference_path in KNOWN_SPEAKERS.items()]
    raw_transcription = transcribe_with_diarization(AUDIO_FILE, known_speakers)
    real_segments = normalize_segments(raw_transcription)
    if REDACT_REAL_AUDIO:
        real_segments = redact_segments(real_segments)

    moderation_results: dict[str, Any] = {}
    if MODERATE_REAL_AUDIO:
        moderation_results["transcript"] = moderate_text(transcript_for_model(real_segments))

    real_intelligence = generate_meeting_intelligence(real_segments)
    real_brief = render_meeting_brief(real_intelligence)
    if MODERATE_REAL_AUDIO:
        moderation_results["meeting_brief"] = moderate_text(real_brief)

    real_report = build_guardrail_report(
        segments=real_segments,
        intelligence=real_intelligence,
        meeting_brief=real_brief,
        redaction_enabled=REDACT_REAL_AUDIO,
        raw_saved=SAVE_RAW_RESPONSE,
        moderation_results=moderation_results,
    )
    write_artifacts(
        REAL_OUTPUT_DIR,
        real_segments,
        real_intelligence,
        real_report,
        raw_payload=raw_transcription if SAVE_RAW_RESPONSE else None,
    )
    print(f"Wrote real-audio artifacts to {REAL_OUTPUT_DIR}")
else:
    print("Skipped real audio run. Set RUN_REAL_AUDIO = True after configuring AUDIO_FILE and OPENAI_API_KEY.")


## 第 9 步：运行确定性的冒烟检查和回归检查

这些检查是确定性的，不会调用 API。请将它们视为针对 notebook 机制的冒烟测试和回归测试套件，而不是对模型质量的评估。

### 冒烟测试检查

第一批检查确认 notebook 写出了所有预期的产物，并将中风险输出转交审核。

### 回归检查

其余断言用于捕获以下方面的回归：schema 可空性、证据引用、演示中缺乏依据的结论、响应边界情况、脱敏以及时间戳格式。


In [ ]:
expected_files = {
    "transcript_segments.json",
    "speaker_labeled_transcript.md",
    "meeting_intelligence.json",
    "meeting_brief.md",
    "guardrail_report.json",
}
assert expected_files.issubset({path.name for path in output_dir.iterdir()})
assert guardrail_report["status"] == "review_required"
assert any(check["name"] == "risk_outputs" and check["status"] == "review" for check in guardrail_report["checks"])
assert any(check["name"] == "evidence_refs" and check["status"] == "pass" for check in guardrail_report["checks"])

action_schema = MEETING_INTELLIGENCE_SCHEMA["schema"]["properties"]["action_items"]["items"]["properties"]
participant_schema = MEETING_INTELLIGENCE_SCHEMA["schema"]["properties"]["participants"]["items"]["properties"]
question_schema = MEETING_INTELLIGENCE_SCHEMA["schema"]["properties"]["explicit_questions"]["items"]["properties"]
assert "null" in action_schema["due_date_or_trigger"]["type"]
assert "null" in action_schema["owner_speaker"]["type"]
assert "null" in participant_schema["inferred_role"]["type"]
assert "null" in question_schema["directed_to_speaker"]["type"]

assert demo_run["segments"][0].segment_id == "seg_001"
assert demo_run["intelligence"]["decisions"] == []
assert demo_run["intelligence"]["action_items"][0]["due_date_or_trigger"] is None
assert demo_run["intelligence"]["action_items"][0]["evidence_refs"][0]["segment_id"] == "seg_005"
assert demo_run["intelligence"]["explicit_questions"]
assert demo_run["intelligence"]["suggested_follow_ups"]
assert validate_evidence_refs(demo_run["intelligence"], demo_run["segments"]) == []
assert "structured outputs" not in demo_run["intelligence"]["follow_up_email"]["body"].lower()
assert "`seg_005`" in demo_run["meeting_brief"]
assert "_Not specified._" in demo_run["meeting_brief"]

broken_intelligence = json.loads(json.dumps(demo_run["intelligence"]))
broken_intelligence["action_items"][0]["evidence_refs"] = [{"segment_id": "seg_999", "quote": "I will send a prototype"}]
assert validate_evidence_refs(broken_intelligence, demo_run["segments"])

broken_intelligence = json.loads(json.dumps(demo_run["intelligence"]))
broken_intelligence["action_items"][0]["evidence_refs"] = [{"segment_id": "seg_005", "quote": "I will send the contract tomorrow"}]
assert validate_evidence_refs(broken_intelligence, demo_run["segments"])

try:
    response_output_text_or_raise({"status": "incomplete", "incomplete_details": {"reason": "max_output_tokens"}})
    raise AssertionError("Expected incomplete response to raise")
except RuntimeError as exc:
    assert "incomplete" in str(exc)

try:
    response_output_text_or_raise({"status": "completed", "output": [{"content": [{"type": "refusal", "refusal": "Cannot comply."}]}]})
    raise AssertionError("Expected refusal response to raise")
except RuntimeError as exc:
    assert "refusal" in str(exc).lower()

try:
    response_output_text_or_raise({"status": "completed", "output_text": ""})
    raise AssertionError("Expected empty response to raise")
except RuntimeError as exc:
    assert "empty" in str(exc).lower()

try:
    parse_meeting_intelligence_json("not json")
    raise AssertionError("Expected invalid JSON to raise")
except RuntimeError as exc:
    assert "invalid JSON" in str(exc)

redacted = redact_segments([
    Segment("seg_test", "Customer", 0.0, 3.0, "Email me at alex@example.com or call 415-555-0100.")
])
assert redacted[0].segment_id == "seg_test"
assert redacted[0].text == "Email me at [email] or call [phone]."
assert format_timestamp(6000) == "100:00.000"

with tempfile.NamedTemporaryFile(suffix=".wav") as oversized_audio:
    oversized_audio.truncate(MAX_AUDIO_UPLOAD_BYTES + 1)
    oversized_audio.flush()
    try:
        transcribe_with_diarization(Path(oversized_audio.name), [])
        raise AssertionError("Expected oversized audio to raise")
    except ValueError as exc:
        assert "25 MB" in str(exc)

with tempfile.NamedTemporaryFile(suffix=".wav") as tiny_audio:
    try:
        transcribe_with_diarization(Path(tiny_audio.name), [], request_timeout_seconds=0)
        raise AssertionError("Expected invalid timeout to raise")
    except ValueError as exc:
        assert "positive" in str(exc)

streamed_transcription = collect_streamed_transcription([
    {"type": "transcript.text.segment", "id": "seg_stream", "speaker": "A", "start": 0.0, "end": 1.0, "text": "Hello"},
    {"type": "transcript.text.done", "text": "Hello", "usage": {"total_tokens": 1}},
])
streamed_segments = normalize_segments(streamed_transcription)
assert streamed_segments[0].segment_id == "seg_stream"
assert streamed_segments[0].speaker == "A"
assert streamed_transcription["usage"]["total_tokens"] == 1

print("Notebook demo validation passed")


## 第 10 步：运行确定性评估

本节针对带标签的演示 fixture 运行一个小型的确定性评估。它仍然不是一个全面的生产评估，但展示了如何在加入模型评分的判断之前，先用可复现的规则对提取质量进行评分。

下面的评分器衡量行动项的精确率/召回率、明确问题的精确率/召回率、缺乏依据的决策、可空的未知字段以及证据引用的有效性。在生产环境中，请用更大的带标签数据集替换 `GOLD_EVAL_LABELS`，并在每个示例上运行相同的评分器。


In [ ]:
GOLD_EVAL_LABELS: dict[str, Any] = {
    "action_items": [
        {
            "owner_speaker": "Solutions Engineer",
            "task_contains": ["send a prototype", "speaker-aware transcripts", "redaction pass", "crm sync"],
            "due_date_or_trigger": None,
            "evidence_segment_ids": ["seg_005"],
        }
    ],
    "explicit_questions": [
        {
            "question_contains": ["support handoff", "break down"],
            "asked_by_speaker": "Solutions Engineer",
            "directed_to_speaker": "Customer",
            "evidence_segment_ids": ["seg_001"],
        }
    ],
    "decisions": [],
}


def normalize_for_eval(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip().casefold()


def evidence_ref_segment_ids(item: dict[str, Any]) -> set[str]:
    return {str(ref.get("segment_id", "")) for ref in item.get("evidence_refs", []) if isinstance(ref, dict)}


def contains_all_fragments(text: Any, fragments: list[str]) -> bool:
    normalized = normalize_for_eval(text)
    return all(normalize_for_eval(fragment) in normalized for fragment in fragments)


def action_item_matches_label(item: dict[str, Any], label: dict[str, Any]) -> bool:
    return (
        item.get("owner_speaker") == label.get("owner_speaker")
        and item.get("due_date_or_trigger") == label.get("due_date_or_trigger")
        and contains_all_fragments(item.get("task"), label.get("task_contains", []))
        and set(label.get("evidence_segment_ids", [])).issubset(evidence_ref_segment_ids(item))
    )


def explicit_question_matches_label(item: dict[str, Any], label: dict[str, Any]) -> bool:
    return (
        item.get("asked_by_speaker") == label.get("asked_by_speaker")
        and item.get("directed_to_speaker") == label.get("directed_to_speaker")
        and contains_all_fragments(item.get("question"), label.get("question_contains", []))
        and set(label.get("evidence_segment_ids", [])).issubset(evidence_ref_segment_ids(item))
    )


def precision_recall(predicted: list[dict[str, Any]], labels: list[dict[str, Any]], matcher) -> dict[str, Any]:
    matched_label_indexes: set[int] = set()
    matched_predictions = 0

    for item in predicted:
        for index, label in enumerate(labels):
            if index in matched_label_indexes:
                continue
            if matcher(item, label):
                matched_label_indexes.add(index)
                matched_predictions += 1
                break

    precision = matched_predictions / len(predicted) if predicted else (1.0 if not labels else 0.0)
    recall = len(matched_label_indexes) / len(labels) if labels else 1.0
    return {
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "matched_predictions": matched_predictions,
        "predicted_count": len(predicted),
        "label_count": len(labels),
    }


def evidence_ref_count(intelligence: dict[str, Any]) -> int:
    return sum(len(refs) for _, refs in iter_evidence_refs(intelligence) if isinstance(refs, list)) + len(intelligence.get("notable_quotes", []))


def run_deterministic_evals(
    intelligence: dict[str, Any],
    segments: list[Segment],
    labels: dict[str, Any],
) -> dict[str, Any]:
    action_item_scores = precision_recall(
        intelligence.get("action_items", []),
        labels.get("action_items", []),
        action_item_matches_label,
    )
    explicit_question_scores = precision_recall(
        intelligence.get("explicit_questions", []),
        labels.get("explicit_questions", []),
        explicit_question_matches_label,
    )
    evidence_problems = validate_evidence_refs(intelligence, segments)
    total_evidence_refs = evidence_ref_count(intelligence)
    valid_evidence_ref_rate = (
        round((total_evidence_refs - len(evidence_problems)) / total_evidence_refs, 3)
        if total_evidence_refs
        else 1.0
    )
    action_items = intelligence.get("action_items", [])
    nullable_due_date_rate = (
        round(sum(1 for item in action_items if item.get("due_date_or_trigger") is None) / len(action_items), 3)
        if action_items
        else 1.0
    )
    unsupported_decision_count = len(intelligence.get("decisions", [])) if not labels.get("decisions") else 0

    pass_conditions = [
        action_item_scores["precision"] == 1.0,
        action_item_scores["recall"] == 1.0,
        explicit_question_scores["precision"] == 1.0,
        explicit_question_scores["recall"] == 1.0,
        valid_evidence_ref_rate == 1.0,
        nullable_due_date_rate == 1.0,
        unsupported_decision_count == 0,
    ]

    return {
        "status": "pass" if all(pass_conditions) else "review_required",
        "action_items": action_item_scores,
        "explicit_questions": explicit_question_scores,
        "valid_evidence_ref_rate": valid_evidence_ref_rate,
        "evidence_ref_problem_count": len(evidence_problems),
        "unsupported_decision_count": unsupported_decision_count,
        "nullable_due_date_rate": nullable_due_date_rate,
    }


deterministic_eval_report = run_deterministic_evals(
    demo_run["intelligence"],
    demo_run["segments"],
    GOLD_EVAL_LABELS,
)
show_json(deterministic_eval_report, expanded=True)
assert deterministic_eval_report["status"] == "pass"


## 第 11 步：添加可选的 LLM 作为评判者评估

LLM 作为评判者的评估适合用来评估确定性评分器无法完全捕捉的质量，例如摘要的有用性、遗漏的跟进事项，以及简报是否能帮助审核人员。请将其保留为可选项，因为它会调用 API，并且结果可能因评判模型而异。请将其与确定性评分器配合使用，而不是取而代之。

下面的评判者会收到文字稿、结构化输出和一份评分细则。它会返回评分和审核发现。对于默认的无网络 notebook 运行，请保持 `RUN_LLM_JUDGE_EVAL = False`。


In [ ]:
RUN_LLM_JUDGE_EVAL = False
LLM_JUDGE_MODEL = os.getenv("OPENAI_MEETING_INTELLIGENCE_JUDGE_MODEL", DEFAULT_SUMMARY_MODEL)

LLM_JUDGE_SCHEMA: dict[str, Any] = {
    "name": "meeting_intelligence_judge",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "outcome": {"type": "string", "enum": ["pass", "review", "fail"]},
            "overall_score": {"type": "number"},
            "scores": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "grounding": {"type": "number"},
                    "action_item_correctness": {"type": "number"},
                    "completeness": {"type": "number"},
                    "safety_review_readiness": {"type": "number"},
                },
                "required": ["grounding", "action_item_correctness", "completeness", "safety_review_readiness"],
            },
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "area": {"type": "string"},
                        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                        "explanation": {"type": "string"},
                    },
                    "required": ["area", "severity", "explanation"],
                },
            },
        },
        "required": ["outcome", "overall_score", "scores", "findings"],
    },
}


def run_llm_judge_eval(segments: list[Segment], intelligence: dict[str, Any], model: str = LLM_JUDGE_MODEL) -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI()
    transcript = transcript_for_model(segments)
    completion = client.responses.create(
        model=model,
        temperature=0,
        store=False,
        input=[
            {
                "role": "system",
                "content": (
                    "You are judging a meeting-intelligence extraction. Grade only against the transcript. "
                    "Penalize unsupported claims, missing major action items, missing customer risks, incorrect speaker attribution, "
                    "and outputs that are not ready for human review. Return calibrated scores from 0 to 1."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Transcript:\n"
                    f"{transcript}\n\n"
                    "Meeting intelligence JSON:\n"
                    f"{json.dumps(intelligence, indent=2)}"
                ),
            },
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": LLM_JUDGE_SCHEMA["name"],
                "strict": LLM_JUDGE_SCHEMA["strict"],
                "schema": LLM_JUDGE_SCHEMA["schema"],
            }
        },
    )
    return parse_meeting_intelligence_json(response_output_text_or_raise(completion))


if RUN_LLM_JUDGE_EVAL:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running the LLM judge eval.")
    llm_judge_report = run_llm_judge_eval(demo_run["segments"], demo_run["intelligence"])
    show_json(llm_judge_report, expanded=True)
else:
    print("Skipped LLM judge eval. Set RUN_LLM_JUDGE_EVAL = True after configuring OPENAI_API_KEY.")


## 生产加固检查清单

在将示例转化为客户工作流之前，请使用以下检查清单：

| 关注点 | 建议 |
| --- | --- |
| 同意 | 确保通话录音、说话人分离和已知说话人参考在你的产品、策略和地区中是被允许的。 |
| 原始音频保留 | 原始音频只在需要的期间内存储。尽可能持久化规范化后的文字稿片段。 |
| 大型录音 | 在上传之前拒绝超过 25 MB 的文件。对较长的会议进行压缩或拆分，然后在合并结果时保留时间戳偏移。 |
| PII 与 DLP | 将其中包含的电子邮件和电话号码正则表达式仅视为示例。对于敏感或受监管的工作流，请使用经策略批准的 PII/DLP 检测器并进行人工审核。 |
| 说话人参考 | 将参考片段视为敏感数据。尽量少存储，静态加密，在不再需要时轮换/删除。 |
| 证据 | 要求决策、风险和行动项附带结构化证据引用。验证每条引用都指向真实的片段 ID 和引语。 |
| 人工审核 | 将高风险摘要、合规承诺、定价声明或合同条款转交审核。 |
| 内容审核 | 当笔记可能包含不安全内容时，使用 Moderation API 进行有害内容分类。将隐私和合规检查分开处理。 |
| 重试行为 | 对暂时性的 API 错误使用退避策略重试。使用幂等键避免重复写入下游 CRM。 |
| 可观测性 | 记录模型名称、提示词版本、schema 版本、音频时长、延迟、脱敏状态以及审核人员的决定。 |
| 评估 | 每周抽样通话。跟踪说话人归属准确率、行动项精确率以及缺乏依据结论的比例。 |


## 面向生产的评估指导

上面的确定性评估有意保持小规模：它证明了评分模式在带标签的 fixture 上可行。生产团队在将输出写入下游系统之前，应将其扩展为一个有代表性的评估集。从一小组已获同意的录音或文字稿 fixture 开始，创建经人工审核的标签，并保留一个留出集，用于在提示词、schema 或模型变化时进行回归测试。

| 领域 | 示例指标 |
| --- | --- |
| 说话人归属 | 说话人标签准确率、说话人分离错误率、说话人轮次边界准确率、已知说话人匹配率。 |
| 文字稿依据 | 引语精确度、时间戳正确性、证据引用有效性、缺乏依据结论的比例。 |
| 结构化提取 | 行动项、决策、风险、明确问题、建议的跟进事项和客户需求的精确率与召回率。 |
| 安全与隐私 | PII 脱敏召回率、内容审核标记召回率、审核误报率、原始音频保留合规性。 |
| 工作流影响 | CRM 更新耗时、审核人员推翻率、跟进完成率、续约或升级风险的检测延迟。 |

一个有用的初始评估很简单：请审核人员将每个提取出的行动项标记为正确、部分正确、缺乏依据或在输出中缺失。跟踪生成条目的精确率，以及相对于人工标注金标准集的召回率。对于引语和证据，优先使用与文字稿片段文本进行精确匹配或近似精确匹配的检查，以免听起来有用却缺乏依据的摘要蒙混过关。LLM 作为评判者可以帮助评估有用性和完整性，但请将确定性的依据检查保留在流程中，因为它们更容易复现。


## 后续步骤

你可以将同一流水线用于：

- 季度业务回顾之后的客户成功交接。
- 注重问责和原话引用的支持升级。
- 为 CRM 提供后续步骤的销售探索电话。
- 每位面试官都需要有出处笔记的招聘面试复盘。
- 需要更严格审核和保留控制的医疗或金融服务工作流。

对于实时场景，可以使用 Realtime 提供通话中的体验，而当你需要持久、有证据支撑的会议智能时，仍然运行这条通话后的说话人分离流水线。

有用的文档：

- [音频与语音指南](https://developers.openai.com/api/docs/guides/audio)
- [语音转文本与说话人分离](https://developers.openai.com/api/docs/guides/speech-to-text)
- [结构化输出](https://developers.openai.com/api/docs/guides/structured-outputs)
- [内容审核](https://developers.openai.com/api/docs/guides/moderation)
- [安全最佳实践](https://developers.openai.com/api/docs/guides/safety-best-practices)
- [Realtime 指南](https://developers.openai.com/api/docs/guides/realtime)